## Example 1 — the import, the version and the list of operations

In [ ]:
import collections, importlib.metadata
import b12x

# b12x has no __version__ attribute: the installed metadata is the version of record.
print("b12x", importlib.metadata.version("b12x"))
operations = {f"{op.group}.{op.name}": op for op in b12x.list_ops()}  # keyed by "<group>.<name>"
names = sorted(operations)
assert len(names) >= 4, names
per_group = collections.Counter(op.group for op in operations.values())
assert {"gemm", "attention", "moe", "quantization"} <= set(per_group), per_group
for wanted in ("gemm.blockscaled", "moe.fused_moe", "attention.paged", "quantization.nvfp4"):
    assert wanted in operations, wanted
assert b12x.find_op("gemm.blockscaled") == operations["gemm.blockscaled"]  # lookup by name agrees with the list
print(f"OK -- {len(names)} operations in {len(per_group)} groups:", dict(per_group))

## Example 2 — the modules vLLM imports load, and the installed CuTe DSL meets b12x's minimum

In [ ]:
import importlib, importlib.metadata
import b12x._lib.gating as gating

# vLLM's `vllm/utils/b12x.py` imports these six and treats ANY ImportError as "b12x is not installed".
VLLM_IMPORTS = ("b12x.attention.paged", "b12x.gemm.blockscaled", "b12x._lib.intrinsics",
                "b12x.gemm.mxfp8_linear", "b12x.gemm.tensor_fp8_linear", "b12x.moe.fused_moe")
for name in VLLM_IMPORTS:
    module = importlib.import_module(name)  # a failure here reds the cell
    assert module.__name__ == name
assert gating.has_cutlass_dsl(), "b12x does not see the CuTe DSL"
gating.require_cutlass_dsl()  # raises if the installed DSL is older than b12x's floor
installed = importlib.metadata.version("nvidia-cutlass-dsl")
floor = gating.MIN_CUTLASS_DSL
as_numbers = lambda version: tuple(int(part) for part in version.split(".")[:3])
assert gating.cutlass_dsl_version() == installed and as_numbers(installed) >= as_numbers(floor), (installed, floor)
print(f"OK -- the {len(VLLM_IMPORTS)} modules vLLM imports load; CuTe DSL {installed} is at or above b12x's floor {floor}")

## Example 3 — what each operation says about itself

In [ ]:
import collections
import b12x

operations = {f"{op.group}.{op.name}": op for op in b12x.list_ops()}
architectures = sorted({arch for op in operations.values() for arch in op.archs})
assert all(arch.startswith("sm12") for arch in architectures), architectures  # one GPU family, nothing else
for op in operations.values():
    assert op.archs and op.dtypes and op.status == "experimental", op
    assert op.api_style and op.entry_points, op
styles = collections.Counter(op.api_style for op in operations.values())
without_probe = sorted(name for name, op in operations.items() if "is_supported" not in op.entry_points)
print("API styles:", dict(styles), "| operations without is_supported():", without_probe)
for wanted in ("gemm.blockscaled", "moe.fused_moe", "attention.paged", "quantization.nvfp4"):
    assert "is_supported" in operations[wanted].entry_points, wanted
paged = b12x.find_op("attention.paged")
assert {"plan", "bind", "run"} <= set(paged.entry_points) and "fp8_e4m3" in paged.dtypes, paged
try:
    b12x.find_op("gemm.no_such_op")
except KeyError as error:
    refused = type(error).__name__
else:
    raise AssertionError("an unknown operation name was accepted")
assert not b12x.kernel_resolution_frozen()
print(f"OK -- all {len(operations)} operations target {architectures}; an unknown operation name raises {refused}")

## Example 4 — an FP8 matrix multiply on the GPU, checked against torch

In [ ]:
import torch
from b12x.gemm import blockscaled
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
if major != 12:
    assert not blockscaled.is_supported(), f"b12x claims support on compute capability {major}.{minor}"
    print(f"SKIP: b12x has kernels for compute capability 12.x only; this GPU is {major}.{minor}")
else:
    assert blockscaled.is_supported(), "b12x refuses a compute capability 12.x GPU"
    torch.manual_seed(0)
    weight = (torch.randn(256, 512, device="cuda") * 0.5).to(torch.float8_e4m3fn)
    activations = (torch.randn(32, 512, device="cuda") * 0.5).to(torch.float8_e4m3fn)
    packed = blockscaled.pack_weight(weight, torch.tensor(1.0, device="cuda"))  # scalar scale: tensor-scaled FP8
    product = blockscaled.mm(activations, packed)
    reference = activations.float() @ weight.float().T
    assert product.shape == (32, 256) and product.dtype == torch.bfloat16, (product.shape, product.dtype)
    error = ((product.float() - reference).norm() / reference.norm()).item()
    assert error < 0.01, f"relative error {error:.4f}"
    assert torch.equal(product, blockscaled.mm(activations, packed)), "the second call differs from the first"
    print(f"OK -- FP8 x FP8 -> BF16 product of shape {tuple(product.shape)}, relative error {error:.4f}")

## Example 5 — freezing kernel resolution: a new shape is refused instead of compiling mid-request

In [ ]:
import torch
import b12x
from b12x.gemm import blockscaled
from b12x.preparation import PreparedCall
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
if major != 12:
    print(f"SKIP: b12x has kernels for compute capability 12.x only; this GPU is {major}.{minor}")
else:
    torch.manual_seed(0)
    one = torch.tensor(1.0, device="cuda")
    make = lambda rows, cols: (torch.randn(rows, cols, device="cuda") * 0.5).to(torch.float8_e4m3fn)
    warm = blockscaled.pack_weight(make(256, 512), one)
    activations = make(32, 512)
    other_weight = blockscaled.pack_weight(make(128, 384), one)  # packed before the freeze
    other_activations = make(8, 384)
    # Declare the shape this cell serves, prepare it in a session, then freeze the session.
    # compile_workers=0 compiles in this process, so the session holds no compiler pool to drain.
    plan = blockscaled.plan(blockscaled.query_from_call(activations, warm))
    request = plan.request(name="example5", prepare_call=lambda state: PreparedCall(run=lambda: None))
    with b12x.PreparationSession(autotune=False, compile_workers=0) as session:
        session.prepare([request])
        session.freeze()
        assert b12x.kernel_resolution_frozen()
        product = blockscaled.mm(activations, warm, plan=plan)  # the prepared shape still runs
        assert product.shape == (32, 256) and product.dtype == torch.bfloat16, (product.shape, product.dtype)
        try:
            blockscaled.mm(other_activations, other_weight)  # a shape nobody prepared
        except RuntimeError as error:  # KernelResolutionFrozenError is a RuntimeError too
            refused = type(error).__name__
        else:
            raise AssertionError("a new shape was prepared while kernel resolution was frozen")
    assert not b12x.kernel_resolution_frozen(), "closing the session did not lift the freeze"
    print(f"OK -- frozen: the prepared shape runs, a new one raises {refused}; closing the session unfreezes")